In [1]:
!pip install numpy pandas

In [2]:
import random
import time
import numpy as np
from itertools import combinations
import os
os.chdir('/Users/mohamedshiyas/Downloads/minhash/')

# load all four documents
doc_files = ['D1.txt', 'D2.txt', 'D3.txt', 'D4.txt']
documents = {}
for fname in doc_files:
    with open(fname, 'r') as f:
        documents[fname.replace('.txt', '')] = f.read().strip()

# quick sanity check
for name, text in documents.items():
    print(f"{name}: {len(text)} chars, first 80: {text[:80]}...")

D1: 1749 chars, first 80: apple ceo tim cook is spending some time in canada this week and yesterday he at...
D2: 1747 chars, first 80: apple ceo tim cook is spending some time in canada this week and yesterday atten...
D3: 2132 chars, first 80: as part of his one day tour of canada yesterday tim cook offered an interview to...
D4: 1435 chars, first 80: president trump who warned as a candidate about the false song of globalism is m...


In [3]:
# ## Question 1: Create k-Grams
#
# We build character 2-grams, character 3-grams, and word 2-grams for each document.
# Only unique k-grams are kept (stored as sets).

# %%
def char_kgrams(text, k):
    """Extract unique character k-grams from text."""
    grams = set()
    for i in range(len(text) - k + 1):
        grams.add(text[i:i+k])
    return grams

def word_kgrams(text, k):
    """Extract unique word k-grams from text."""
    words = text.split()
    grams = set()
    for i in range(len(words) - k + 1):
        gram = tuple(words[i:i+k])
        grams.add(gram)
    return grams

# build all k-gram sets
char2 = {}
char3 = {}
word2 = {}

for name, text in documents.items():
    char2[name] = char_kgrams(text, 2)
    char3[name] = char_kgrams(text, 3)
    word2[name] = word_kgrams(text, 2)

# print sizes
print("=== Number of unique k-grams per document ===")
print(f"{'Doc':<5} {'Char-2':<10} {'Char-3':<10} {'Word-2':<10}")
for name in ['D1', 'D2', 'D3', 'D4']:
    print(f"{name:<5} {len(char2[name]):<10} {len(char3[name]):<10} {len(word2[name]):<10}")

=== Number of unique k-grams per document ===
Doc   Char-2     Char-3     Word-2    
D1    263        765        279       
D2    262        762        278       
D3    269        828        337       
D4    255        698        232       


In [4]:
# ### 1B: Jaccard Similarity for all pairs, all k-gram types

# %%
def jaccard(set_a, set_b):
    """Compute exact Jaccard similarity between two sets."""
    if len(set_a) == 0 and len(set_b) == 0:
        return 1.0
    intersection = len(set_a & set_b)
    union = len(set_a | set_b)
    return intersection / union

doc_names = ['D1', 'D2', 'D3', 'D4']
pairs = list(combinations(doc_names, 2))

print("=== Exact Jaccard Similarities ===\n")
for gram_name, gram_dict in [("Char 2-gram", char2), ("Char 3-gram", char3), ("Word 2-gram", word2)]:
    print(f"--- {gram_name} ---")
    for d1, d2 in pairs:
        j = jaccard(gram_dict[d1], gram_dict[d2])
        print(f"  J({d1}, {d2}) = {j:.4f}")
    print()

=== Exact Jaccard Similarities ===

--- Char 2-gram ---
  J(D1, D2) = 0.9811
  J(D1, D3) = 0.8157
  J(D1, D4) = 0.6444
  J(D2, D3) = 0.8000
  J(D2, D4) = 0.6413
  J(D3, D4) = 0.6530

--- Char 3-gram ---
  J(D1, D2) = 0.9780
  J(D1, D3) = 0.5804
  J(D1, D4) = 0.3051
  J(D2, D3) = 0.5680
  J(D2, D4) = 0.3059
  J(D3, D4) = 0.3121

--- Word 2-gram ---
  J(D1, D2) = 0.9408
  J(D1, D3) = 0.1823
  J(D1, D4) = 0.0302
  J(D2, D3) = 0.1737
  J(D2, D4) = 0.0303
  J(D3, D4) = 0.0161



In [5]:
# ### Observations
#
# From the k-gram analysis, we can see that D1 and D2 are almost the same 
# document with very minor differences. Their Jaccard similarity is very 
# high across all three k-gram types (0.98 for char-2, 0.97 for char-3, 
# 0.94 for word-2). This makes sense because both documents are basically 
# the same article about Tim Cook's visit to Canada with only a few words 
# changed here and there.
#
# D4 is about a completely different topic (President Trump at Davos), so 
# its similarity with D1, D2 and D3 is quite low. D3 is also about Tim 
# Cook but written in a different style, so it shares moderate similarity 
# with D1 and D2.
#
# One interesting thing to note is that as we move from character 2-grams 
# to character 3-grams to word 2-grams, the similarity values between 
# different-topic documents keep dropping. This happens because character 
# level grams capture common letter patterns (like "th", "he", "in") that 
# appear in any English text, whereas word level grams are more specific 
# to the actual content. So word 2-grams give us a better picture of 
# whether two documents are actually talking about the same thing.

In [6]:
# ## Question 2: Min-Hashing
#
# We use hash functions of the form: h(x) = (a*x + b) % p
# where p is a large prime and a,b are random.
# Each k-gram gets mapped to an integer first, then we hash it.

# %%
# first, build a universal vocabulary of 3-grams across all docs
# and assign each gram a unique integer id
all_3grams = set()
for name in doc_names:
    all_3grams |= char3[name]

gram_to_id = {g: idx for idx, g in enumerate(sorted(all_3grams))}
print(f"Total unique 3-grams across all docs: {len(gram_to_id)}")

# convert each document's 3-gram set into a set of integer ids
doc_ids = {}
for name in doc_names:
    doc_ids[name] = set(gram_to_id[g] for g in char3[name])

# %%
def generate_hash_funcs(t, p=10007):
    """Generate t random hash functions of form h(x) = (a*x + b) % p."""
    funcs = []
    seen = set()
    while len(funcs) < t:
        a = random.randint(1, p - 1)
        b = random.randint(0, p - 1)
        if (a, b) not in seen:
            seen.add((a, b))
            funcs.append((a, b))
    return funcs, p

def compute_minhash_signature(doc_id_set, hash_funcs, p):
    """
    Compute the minhash signature for a document.
    For each hash function, find the minimum hash value
    over all elements in the document's set.
    """
    sig = []
    for a, b in hash_funcs:
        min_val = float('inf')
        for x in doc_id_set:
            h = (a * x + b) % p
            if h < min_val:
                min_val = h
        sig.append(min_val)
    return sig

def approx_jaccard(sig1, sig2):
    """Estimate Jaccard similarity from two minhash signatures."""
    assert len(sig1) == len(sig2)
    matches = sum(1 for a, b in zip(sig1, sig2) if a == b)
    return matches / len(sig1)

Total unique 3-grams across all docs: 1301


In [7]:
# ### 2A: MinHash signatures for D1 & D2 with t = 20, 60, 150, 300, 600

# %%
t_values = [20, 60, 150, 300, 600]

# get the exact jaccard for reference
exact_j_d1d2 = jaccard(char3['D1'], char3['D2'])
print(f"Exact Jaccard (3-gram) for D1,D2: {exact_j_d1d2:.4f}\n")

print(f"{'t':<8} {'Approx J(D1,D2)':<20} {'Error':<10} {'Time (s)':<10}")
print("-" * 50)

random.seed(42)  # for reproducibility

for t in t_values:
    start = time.time()
    hfuncs, p = generate_hash_funcs(t)
    sig_d1 = compute_minhash_signature(doc_ids['D1'], hfuncs, p)
    sig_d2 = compute_minhash_signature(doc_ids['D2'], hfuncs, p)
    approx_j = approx_jaccard(sig_d1, sig_d2)
    elapsed = time.time() - start
    err = abs(approx_j - exact_j_d1d2)
    print(f"{t:<8} {approx_j:<20.4f} {err:<10.4f} {elapsed:<10.4f}")

Exact Jaccard (3-gram) for D1,D2: 0.9780

t        Approx J(D1,D2)      Error      Time (s)  
--------------------------------------------------
20       0.9500               0.0280     0.0019    
60       0.9667               0.0113     0.0058    
150      0.9667               0.0113     0.0146    
300      0.9800               0.0020     0.0282    
600      0.9900               0.0120     0.0561    


In [8]:
# ### 2B: What is a good value for t?
#
# **Observation:** As t increases, the approximation gets closer to the exact
# Jaccard similarity, but computation time also increases roughly linearly.
#
# From the results above, t=150 or t=300 tends to give a good balance —
# the error drops significantly compared to t=20 or t=60, while the time
# is still very manageable. Beyond t=300, the improvement in accuracy is
# marginal.
#
# **Conclusion:** t=150 seems like a sweet spot for this dataset size.
# It gives accuracy within ~0.02-0.03 of the true value and runs fast.

print("=== Stability experiment (10 runs each) ===\n")
print(f"{'t':<8} {'Mean Error':<15} {'Std Error':<15} {'Mean Time':<10}")
print("-" * 50)

for t in t_values:
    errors = []
    times = []
    for _ in range(10):
        start = time.time()
        hfuncs, p = generate_hash_funcs(t)
        sig1 = compute_minhash_signature(doc_ids['D1'], hfuncs, p)
        sig2 = compute_minhash_signature(doc_ids['D2'], hfuncs, p)
        aj = approx_jaccard(sig1, sig2)
        elapsed = time.time() - start
        errors.append(abs(aj - exact_j_d1d2))
        times.append(elapsed)
    print(f"{t:<8} {np.mean(errors):<15.4f} {np.std(errors):<15.4f} {np.mean(times):<10.4f}")

=== Stability experiment (10 runs each) ===

t        Mean Error      Std Error       Mean Time 
--------------------------------------------------
20       0.0350          0.0216          0.0019    
60       0.0126          0.0090          0.0054    
150      0.0091          0.0065          0.0139    
300      0.0111          0.0061          0.0270    
600      0.0044          0.0028          0.0538    


In [9]:
# ## Question 3: LSH
#
# We use t=160 hash functions and want to find pairs with J > τ = 0.7

# %% [markdown]
# ### 3A: Finding optimal b and r
#
# We need b * r = t = 160. The S-curve is f(s) = 1 - (1 - s^b)^r.
# We want good separation at τ = 0.7, meaning:
# - High probability for pairs with J >= 0.7
# - Low probability for pairs with J < 0.7
#
# The threshold of the S-curve is approximately at s* = (1/r)^(1/b).
# We want s* ≈ 0.7.

# %%
# try all valid (b, r) combos where b*r = 160
def s_curve(s, b, r):
    return 1.0 - (1.0 - s**b)**r

t_lsh = 160
tau = 0.7

print(f"{'b':<6} {'r':<6} {'f(0.7)':<10} {'f(0.5)':<10} {'f(0.9)':<10} {'threshold':<12}")
print("-" * 56)

candidates = []
for b in range(1, t_lsh + 1):
    if t_lsh % b == 0:
        r = t_lsh // b
        f_tau = s_curve(tau, b, r)
        f_low = s_curve(0.5, b, r)
        f_high = s_curve(0.9, b, r)
        threshold = (1.0 / r) ** (1.0 / b)
        candidates.append((b, r, f_tau, f_low, f_high, threshold))
        print(f"{b:<6} {r:<6} {f_tau:<10.4f} {f_low:<10.4f} {f_high:<10.4f} {threshold:<12.4f}")

# pick the (b, r) whose threshold is closest to tau
best = min(candidates, key=lambda x: abs(x[5] - tau))
b_best, r_best = best[0], best[1]
print(f"\nBest (b, r) = ({b_best}, {r_best}) with threshold ≈ {best[5]:.4f}")
print(f"f(0.7) = {best[2]:.4f}, f(0.5) = {best[3]:.4f}, f(0.9) = {best[4]:.4f}")

b      r      f(0.7)     f(0.5)     f(0.9)     threshold   
--------------------------------------------------------
1      160    1.0000     1.0000     1.0000     0.0063      
2      80     1.0000     1.0000     1.0000     0.1118      
4      40     1.0000     0.9243     1.0000     0.3976      
5      32     0.9972     0.6379     1.0000     0.5000      
8      20     0.6950     0.0753     1.0000     0.6877      
10     16     0.3677     0.0155     0.9990     0.7579      
16     10     0.0327     0.0002     0.8712     0.8660      
20     8      0.0064     0.0000     0.6455     0.9013      
32     5      0.0001     0.0000     0.1603     0.9509      
40     4      0.0000     0.0000     0.0578     0.9659      
80     2      0.0000     0.0000     0.0004     0.9914      
160    1      0.0000     0.0000     0.0000     1.0000      

Best (b, r) = (8, 20) with threshold ≈ 0.6877
f(0.7) = 0.6950, f(0.5) = 0.0753, f(0.9) = 1.0000


In [10]:
# ### Observations
#
# For t=160 hash functions and threshold tau=0.7, we found that the best 
# values are b=8 (rows per band) and r=20 (number of bands). This gives 
# a threshold of approximately 0.6877 which is the closest we can get to 
# 0.7 among all valid factor pairs of 160.
#
# With these values, the S-curve gives us f(0.5) = 0.075 and f(0.9) = 1.0. 
# This means pairs with low similarity (0.5) have only 7.5% chance of 
# becoming candidates, while pairs with high similarity (0.9) will almost 
# certainly be detected. The separation at tau=0.7 is decent with 
# f(0.7) = 0.695.

In [11]:
# ### 3B: Probability of each pair being a candidate (using 3-grams)

# %%
# first compute exact jaccard for all pairs using 3-grams
print("=== LSH candidate probabilities (3-grams, t=160) ===\n")
print(f"Using b={b_best}, r={r_best}\n")

print(f"{'Pair':<12} {'Exact J':<12} {'f(J) = P(candidate)':<20}")
print("-" * 45)

for d1, d2 in pairs:
    j = jaccard(char3[d1], char3[d2])
    prob = s_curve(j, b_best, r_best)
    print(f"({d1},{d2})    {j:<12.4f} {prob:<20.6f}")

=== LSH candidate probabilities (3-grams, t=160) ===

Using b=8, r=20

Pair         Exact J      f(J) = P(candidate) 
---------------------------------------------
(D1,D2)    0.9780       1.000000            
(D1,D3)    0.5804       0.228224            
(D1,D4)    0.3051       0.001500            
(D2,D3)    0.5680       0.195880            
(D2,D4)    0.3059       0.001532            
(D3,D4)    0.3121       0.001800            


In [12]:
# ### Observations
#
# The candidate probabilities align well with what we expect:
#
# D1 and D2 have probability 1.0 since their Jaccard similarity (0.978) 
# is way above the threshold. LSH will always identify them as candidates.
#
# D1-D4, D2-D4, and D3-D4 all have very low probabilities (around 0.001 
# to 0.002) because their similarities are around 0.30-0.31, well below 
# the threshold. LSH correctly filters them out.
#
# D1-D3 and D2-D3 have moderate probabilities (0.19-0.23). Their exact 
# similarities are around 0.57-0.58, which is below tau=0.7 but not too 
# far from it. So there is some small chance they get flagged as 
# candidates, which is expected behavior near the transition zone of 
# the S-curve.

In [13]:
# ## Question 4: Min-Hashing on MovieLens 100k

import os
import pandas as pd

# load the ratings data
# format: user_id \t item_id \t rating \t timestamp
data_path = 'ml-100k/u.data'
df = pd.read_csv(data_path, sep='\t', header=None,
                 names=['user', 'movie', 'rating', 'timestamp'])

print(f"Total ratings: {len(df)}")
print(f"Users: {df['user'].nunique()}, Movies: {df['movie'].nunique()}")

# build user -> set of movies
user_movies = {}
for user, group in df.groupby('user'):
    user_movies[user] = set(group['movie'].values)

users = sorted(user_movies.keys())
print(f"\nLoaded {len(users)} users")
print(f"Example - User 1 rated {len(user_movies[1])} movies")

# %% [markdown]
# ### 4A: Exact Jaccard for all user pairs, find pairs with J >= 0.5

# %%
# this takes a while since we have C(943,2) = ~444k pairs
print("Computing exact Jaccard for all user pairs...")
start = time.time()

exact_sim = {}
similar_pairs_exact = []

for i in range(len(users)):
    for j in range(i + 1, len(users)):
        u1, u2 = users[i], users[j]
        s1, s2 = user_movies[u1], user_movies[u2]
        inter = len(s1 & s2)
        union = len(s1 | s2)
        j_val = inter / union if union > 0 else 0
        exact_sim[(u1, u2)] = j_val
        if j_val >= 0.5:
            similar_pairs_exact.append((u1, u2, j_val))

elapsed = time.time() - start
print(f"Done in {elapsed:.2f}s")
print(f"Pairs with J >= 0.5: {len(similar_pairs_exact)}")
print("\nThese pairs are:")
for u1, u2, j_val in sorted(similar_pairs_exact, key=lambda x: -x[2]):
    print(f"  User {u1} & User {u2}: J = {j_val:.4f}")

# %%
# store the exact similar pairs as a set for later comparison
exact_set_05 = set((u1, u2) for u1, u2, _ in similar_pairs_exact)

Total ratings: 100000
Users: 943, Movies: 1682

Loaded 943 users
Example - User 1 rated 272 movies
Computing exact Jaccard for all user pairs...
Done in 2.79s
Pairs with J >= 0.5: 10

These pairs are:
  User 408 & User 898: J = 0.8387
  User 328 & User 788: J = 0.6730
  User 489 & User 587: J = 0.6299
  User 600 & User 826: J = 0.5455
  User 451 & User 489: J = 0.5333
  User 674 & User 879: J = 0.5217
  User 554 & User 764: J = 0.5170
  User 197 & User 826: J = 0.5130
  User 197 & User 600: J = 0.5000
  User 800 & User 879: J = 0.5000


In [14]:
# ### 4B: MinHash approximation with t = 50, 100, 200

# %%
def build_minhash_sigs_movielens(user_movies, users, t, p=10007):
    """Build minhash signatures for all users."""
    hfuncs, p_val = generate_hash_funcs(t, p)
    sigs = {}
    for u in users:
        sig = []
        for a, b in hfuncs:
            min_h = float('inf')
            for movie in user_movies[u]:
                h = (a * movie + b) % p_val
                if h < min_h:
                    min_h = h
            sig.append(min_h)
        sigs[u] = sig
    return sigs

def find_similar_from_sigs(sigs, users, threshold):
    """Find pairs with approx Jaccard >= threshold from signatures."""
    found = []
    for i in range(len(users)):
        for j in range(i + 1, len(users)):
            u1, u2 = users[i], users[j]
            aj = approx_jaccard(sigs[u1], sigs[u2])
            if aj >= threshold:
                found.append((u1, u2))
    return set(found)

# %%
t_vals_ml = [50, 100, 200]
num_runs = 5

print("=== MinHash on MovieLens (5 runs each, threshold=0.5) ===\n")
print(f"Exact pairs with J>=0.5: {len(exact_set_05)}\n")
print(f"{'t':<8} {'Avg FP':<12} {'Avg FN':<12} {'Avg Time(s)':<12}")
print("-" * 46)

# store signatures for Q5 later
saved_sigs = {}

for t in t_vals_ml:
    fp_list = []
    fn_list = []
    time_list = []

    for run in range(num_runs):
        start = time.time()
        sigs = build_minhash_sigs_movielens(user_movies, users, t)
        approx_set = find_similar_from_sigs(sigs, users, 0.5)
        elapsed = time.time() - start

        fp = len(approx_set - exact_set_05)
        fn = len(exact_set_05 - approx_set)

        fp_list.append(fp)
        fn_list.append(fn)
        time_list.append(elapsed)

        # save last run's sigs for Q5
        if run == num_runs - 1:
            saved_sigs[t] = sigs

    print(f"{t:<8} {np.mean(fp_list):<12.1f} {np.mean(fn_list):<12.1f} {np.mean(time_list):<12.2f}")

=== MinHash on MovieLens (5 runs each, threshold=0.5) ===

Exact pairs with J>=0.5: 10

t        Avg FP       Avg FN       Avg Time(s) 
----------------------------------------------
50       119.2        3.0          1.33        
100      22.6         1.6          2.48        
200      7.0          2.2          4.79        


In [15]:
# ### Observations
#
# There are 10 pairs of users in the MovieLens dataset with exact Jaccard 
# similarity of at least 0.5. The most similar pair is User 408 and 
# User 898 with J=0.8387.
#
# When we use MinHash to approximate these similarities:
# - With t=50 hash functions, we get a very high number of false positives 
#   (around 119) because 50 hash functions are not enough to accurately 
#   estimate the similarity. The signatures are too short and many 
#   dissimilar pairs end up looking similar by chance.
# - With t=100, the false positives drop significantly to around 22, and 
#   false negatives are about 1.6.
# - With t=200, false positives come down to around 7 and false negatives 
#   are about 2.2. This is a much better result.
#
# The false negatives remain relatively low across all values of t, meaning 
# MinHash rarely misses a truly similar pair. The main issue with small t 
# is false positives — it wrongly flags many pairs as similar when they 
# are not.

In [16]:
# ## Question 5: LSH on MovieLens
#
# Find candidate pairs with J >= 0.6
# Configurations:
# - r=5, b=10 for t=50
# - r=5, b=20 for t=100
# - r=5, b=40 and r=10, b=20 for t=200

# %%
def lsh_candidates(sigs, users, b, r):
    """
    LSH banding: split each signature into r bands of b rows.
    Two users are candidates if they match in at least one band.
    """
    candidates = set()
    num_bands = r  # number of bands
    band_size = b  # rows per band

    for band_idx in range(num_bands):
        # hash each user's band portion into buckets
        buckets = {}
        start_row = band_idx * band_size
        end_row = start_row + band_size

        for u in users:
            band_slice = tuple(sigs[u][start_row:end_row])
            band_hash = hash(band_slice)
            if band_hash not in buckets:
                buckets[band_hash] = []
            buckets[band_hash].append(u)

        # all pairs in same bucket are candidates
        for bucket_users in buckets.values():
            if len(bucket_users) > 1:
                for i in range(len(bucket_users)):
                    for j in range(i + 1, len(bucket_users)):
                        u1, u2 = bucket_users[i], bucket_users[j]
                        pair = (min(u1, u2), max(u1, u2))
                        candidates.add(pair)

    return candidates

# %%
# compute exact pairs with J >= 0.6
exact_set_06 = set()
for (u1, u2), j_val in exact_sim.items():
    if j_val >= 0.6:
        exact_set_06.add((u1, u2))

# and for J >= 0.8
exact_set_08 = set()
for (u1, u2), j_val in exact_sim.items():
    if j_val >= 0.8:
        exact_set_08.add((u1, u2))

print(f"Exact pairs with J >= 0.6: {len(exact_set_06)}")
print(f"Exact pairs with J >= 0.8: {len(exact_set_08)}")

# %%
# LSH configurations
lsh_configs = [
    (50, 5, 10),    # t=50,  b=5, r=10
    (100, 5, 20),   # t=100, b=5, r=20
    (200, 5, 40),   # t=200, b=5, r=40
    (200, 10, 20),  # t=200, b=10, r=20
]

num_runs = 5

print("=== LSH on MovieLens (5 runs, threshold=0.6) ===\n")
print(f"{'Config':<20} {'Avg FP':<12} {'Avg FN':<12}")
print("-" * 44)

for t, b_lsh, r_lsh in lsh_configs:
    fp_list = []
    fn_list = []

    for run in range(num_runs):
        # build fresh signatures each run
        sigs = build_minhash_sigs_movielens(user_movies, users, t)
        cands = lsh_candidates(sigs, users, b_lsh, r_lsh)

        # for LSH, we need to also check approx similarity of candidates
        # against threshold to get final pairs
        # Actually, LSH gives us candidate pairs. We then check their
        # approximate Jaccard from signatures.
        final_pairs = set()
        for u1, u2 in cands:
            aj = approx_jaccard(sigs[u1], sigs[u2])
            if aj >= 0.6:
                final_pairs.add((u1, u2))

        fp = len(final_pairs - exact_set_06)
        fn = len(exact_set_06 - final_pairs)
        fp_list.append(fp)
        fn_list.append(fn)

    config_str = f"t={t}, b={b_lsh}, r={r_lsh}"
    print(f"{config_str:<20} {np.mean(fp_list):<12.1f} {np.mean(fn_list):<12.1f}")

# %% [markdown]
# ### Threshold = 0.8

# %%
print("=== LSH on MovieLens (5 runs, threshold=0.8) ===\n")
print(f"{'Config':<20} {'Avg FP':<12} {'Avg FN':<12}")
print("-" * 44)

for t, b_lsh, r_lsh in lsh_configs:
    fp_list = []
    fn_list = []

    for run in range(num_runs):
        sigs = build_minhash_sigs_movielens(user_movies, users, t)
        cands = lsh_candidates(sigs, users, b_lsh, r_lsh)

        final_pairs = set()
        for u1, u2 in cands:
            aj = approx_jaccard(sigs[u1], sigs[u2])
            if aj >= 0.8:
                final_pairs.add((u1, u2))

        fp = len(final_pairs - exact_set_08)
        fn = len(exact_set_08 - final_pairs)
        fp_list.append(fp)
        fn_list.append(fn)

    config_str = f"t={t}, b={b_lsh}, r={r_lsh}"
    print(f"{config_str:<20} {np.mean(fp_list):<12.1f} {np.mean(fn_list):<12.1f}")

Exact pairs with J >= 0.6: 3
Exact pairs with J >= 0.8: 1
=== LSH on MovieLens (5 runs, threshold=0.6) ===

Config               Avg FP       Avg FN      
--------------------------------------------
t=50, b=5, r=10      4.4          1.2         
t=100, b=5, r=20     0.6          0.2         
t=200, b=5, r=40     0.0          0.2         
t=200, b=10, r=20    0.0          2.0         
=== LSH on MovieLens (5 runs, threshold=0.8) ===

Config               Avg FP       Avg FN      
--------------------------------------------
t=50, b=5, r=10      0.0          0.2         
t=100, b=5, r=20     0.0          0.0         
t=200, b=5, r=40     0.0          0.0         
t=200, b=10, r=20    0.0          0.0         


In [17]:
# ### Observations
#
# When we change the threshold from 0.6 to 0.8, both false positives and 
# false negatives drop a lot. At threshold 0.6 there are 3 exact pairs, 
# but at 0.8 there is only 1 exact pair (User 408 & User 898, J=0.8387).
# Since this pair has very high similarity, almost all configurations 
# detect it easily, giving 0 FP and 0 FN in most cases.
#
# Comparing the two t=200 configs at threshold 0.6:
# - (b=5, r=40) gives FP=0.0, FN=0.2 — more bands means more chances 
#   for similar pairs to land in the same bucket, so fewer false negatives.
# - (b=10, r=20) gives FP=0.0, FN=2.0 — larger band size means two users 
#   must match on all 10 rows within a band, which is a stricter condition. 
#   So it misses more similar pairs, giving higher false negatives.
#
# In general, increasing the number of bands (r) with smaller band size (b) 
# helps in catching more true pairs but can increase false positives. 
# Larger band size with fewer bands is more strict and reduces false 
# positives but risks missing some similar pairs.